# Commutants and cosets

The *commutant* of a vertex subalgebra $A$ of a vertex algebra $V$ is the subspace of the elements of $V$ whose OPEs with all elements of $A$ are regular,

$$\mathrm{Com}(A, V) = \{ v \in V : a_{(n)} v = 0 \text{ for all } a \in A,\ n \ge 0 \}.$$

It is a vertex subalgebra of $V$ [FZ1992]. In physics it is the *coset* $V/A$ of the coset construction [GKO1985, GKO1986]: if $V$ and $A$ have conformal vectors $T_V$ and $T_A$ (with $T_A$ also conformal for the action of $A$ on $V$), then $T_V - T_A$ is a conformal vector of the commutant, of central charge $c_V - c_A$.

The library computes commutants as follows:

* `V.commutant(A)`, with the alias `V.coset(A)`: here `A` is a vertex subalgebra of `V` (see `06_subalgebras_ideals_quotients.ipynb`) or a list of elements generating one. `S.commutant()` is the commutant of a subalgebra `S` in its ambient algebra.
* Only the *generators* of `A` are tested. For a fixed $v \in V$, the elements $a$ with $a_{(n)}v = 0$ for all $n \ge 0$ form a subalgebra, so $v$ commutes with $A$ as soon as it commutes with its generators. Membership `v in C` is therefore a finite test, and `A` itself is never built.
* `C.weight_space(W)` is the kernel of the modes $a_{(n)}$, $n \ge 0$, of the generators $a$ of $A$ on the weight space $V_W$, with a canonical basis. It is exact.
* `C.strong_generators_up_to(W)` gives strong generators of the commutant, found weight by weight: up to weight `W` every weight space is spanned by their normally ordered monomials. Nothing is claimed above `W` (section 1).
* `V.commutant(A, conformal_vector='coset')`, or `C.with_conformal_vector('coset')`, gives the commutant the conformal vector $T_V - T_A$. For this, `A` must be a subalgebra constructed with its conformal vector $T_A$.
* `C.commutant(max_weight=k)` is the commutant of the subalgebra generated by the strong generators of `C` of weight $\le k$: the *double commutant* $\mathrm{Com}(\mathrm{Com}(A, V), V)$ when `C` is strongly generated in weights $\le k$.

The notebook works through two classical examples:

* **GKO** [GKO1986]: $\mathrm{Com}(L_2(\mathfrak{sl}_2), L_1(\mathfrak{sl}_2) \otimes L_1(\mathfrak{sl}_2))$ is the Ising model $L(\frac12, 0)$;
* **Kazama–Suzuki** [KS1989]: the commutant of a Heisenberg field in $V^k(\mathfrak{sl}_2) \otimes bc$ is the $N = 2$ superconformal algebra of central charge $c = 3k/(k+2)$, for generic level $k$.

**Prerequisites:** `01_getting_started.ipynb`, `05_lattice_vertex_algebras.ipynb` (for section 2), `06_subalgebras_ideals_quotients.ipynb`.

**Elsewhere:** W-algebras are computed as kernels of a differential, a kind of commutant, in `08_w_algebras.ipynb`, section 7.

**Contents**

1. Exact weight spaces, strong generators up to a weight
2. GKO: the Ising model as a coset
3. Kazama–Suzuki: the $N = 2$ superconformal algebra
4. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import time                    # to time the computations of weight spaces

## 1. Exact weight spaces, strong generators up to a weight

Unlike a subalgebra (`06_subalgebras_ideals_quotients.ipynb`, sections 1 and 2), a commutant is not built by a closure, so the library does not know a complete set of strong generators for it. What it knows exactly are the weight spaces: `C.weight_space(n)` is the kernel of the modes $a_{(m)}$, $m \ge 0$, of the generators of `A` on $V_n$, in every weight $n$ that is asked for. Strong generators are found weight by weight, and `C.strong_generators_up_to(W)` returns elements whose normally ordered monomials span every weight space of the commutant up to weight `W`. Nothing is claimed above `W`: a new strong generator could appear in a higher weight, and the method carries this limit in its name.

The smallest example is two free bosons $a$, $b$, with $a(z)a(w) \sim b(z)b(w) \sim 1/(z-w)^2$ and regular $a(z)b(w)$, and the commutant of the diagonal field $a + b$. It contains $b - a$, since $(a+b)(z)(b-a)(w) \sim (1 - 1)/(z-w)^2 = 0$. The cell below shows that up to weight 4 this one field strongly generates the commutant, and prints the dimensions of its weight spaces: the numbers of partitions, as for a single free boson. (That $b - a$ generates the commutant in every weight is a statement about all weights, which the computation does not make.)

In [2]:
H2 = UniversalVertexAlgebra(QQ, ['a', 'b'], {('a', 'a'): [0, 1], ('b', 'b'): [0, 1]},
                            degrees={'a': 1, 'b': 1})
a, b = H2.gens()
C = H2.commutant([a + b])
C.strong_generators_up_to(4), [C.weight_space(n).dimension() for n in range(6)]

([-a + b], [1, 1, 2, 3, 5, 7])

## 2. GKO: the Ising model as a coset

**The ambient algebra.** By the Frenkel–Kac construction, $L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra of the root lattice $A_1$ (see `05_lattice_vertex_algebras.ipynb`, sections 1 and 2). So $V = L_1(\mathfrak{sl}_2) \otimes L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra of $A_1 \oplus A_1$, with Gram matrix $\mathrm{diag}(2, 2)$, two Heisenberg currents $J_0, J_1$ and lattice currents $e^{q}$, $q \in \mathbb Z^2$. Its conformal vector is the sum of the two Sugawara vectors, $T_V = \frac14 {:}J_0J_0{:} + \frac14 {:}J_1J_1{:}$, of central charge $c = 2$.

**The diagonal $\mathfrak{sl}_2$.** The sums $e = e^{\alpha_0} + e^{\alpha_1}$ and $f = e^{-\alpha_0} + e^{-\alpha_1}$ of the currents of the two factors generate the diagonal affine $\mathfrak{sl}_2$. The OPE of $e$ and $f$ gives $h = e_{(0)}f = J_0 + J_1$ and $e_{(1)}f = 2$, so the level is $1 + 1 = 2$. The cell below builds $V$ and prints $h$, the OPEs $e(z)f(w)$ and $h(z)e(w)$, and the central charge.

In [3]:
L = LatticeVertexAlgebra(QQ, matrix([[2, 0], [0, 2]]))
J0, J1 = L.heisenberg_currents()
L = L.with_conformal_vector(J0.no(J0) / 4 + J1.no(J1) / 4)
J0, J1 = L.heisenberg_currents()
e = L.e((1, 0)) + L.e((0, 1))
f = L.e((-1, 0)) + L.e((0, -1))
h = e.nproduct(f, 0)
h, e.ope(f), h.ope(e), L.central_charge()

(J0 + J1, [J0 + J1, 2], [2*e^(0,1) + 2*e^(1,0)], 2)

The subalgebra $A$ generated by $e$ and $f$ is the diagonal $L_2(\mathfrak{sl}_2)$. Its closure (`06_subalgebras_ideals_quotients.ipynb`, section 1) adds $h = e_{(0)}f$ as a strong generator. It is given the Sugawara vector of the diagonal currents at $k = 2$,
$$T_A = \frac{1}{2(k+2)}\Big(\tfrac12 {:}hh{:} + {:}ef{:} + {:}fe{:}\Big),$$
of central charge $3k/(k+2) = \frac32$. The cell below prints the strong generators and the central charge.

In [4]:
A = L.subalgebra([e, f], names=['e', 'f'], conformal_vector=(h.no(h) / 2 + e.no(f) + f.no(e)) / 8)
A.strong_generator_info(), A.central_charge()

([('e', 1, 0, 'given'), ('f', 1, 0, 'given'), ('S0', 1, 0, 'e_(0) f')], 3/2)

**The coset.** `L.coset(A)`, `L.commutant([e, f])` and `A.commutant()` are the same object, since the commutant depends only on the generators of $A$. The first cell below checks this and prints the commutant.

Membership is the finite test "all poles of $e(z)v(w)$ and $f(z)v(w)$ vanish", and needs no weight spaces. The second cell tests four elements: $e$ and $h$ (in $A$, not in the coset), $T_V$ (not in the coset: its OPEs with $e$ and $f$ have the poles of a conformal vector), and $T_V - T_A$, which is in the coset.

In [5]:
C = L.coset(A)
C is L.commutant([e, f]), C is A.commutant(), C

(True,
 True,
 Commutant of the vertex subalgebra generated by [e^(0,1) + e^(1,0), e^(-1,0) + e^(0,-1)] in Lattice vertex algebra of rank 2 with Gram matrix [[2, 0], [0, 2]] over Rational Field)

In [6]:
TV, TA = L.conformal_vector(), A.conformal_vector()
e in C, h in C, TV in C, TV - TA in C

(False, False, False, True)

**Weight spaces.** The weight spaces of the commutant are the kernels of the modes $e_{(n)}, f_{(n)}$, $n \ge 0$, on the weight spaces of $V$. GKO's theorem [GKO1986] says that the coset is the vertex algebra $L(\frac12, 0)$ of the Ising model [BPZ1984], whose character is the even part of the character of a free Majorana fermion,
$$\operatorname{ch} L(\tfrac12, 0) = \frac12\Big(\prod_{n \ge 1}(1 + q^{n-1/2}) + \prod_{n \ge 1}(1 - q^{n-1/2})\Big).$$
The cell below computes the dimensions of the weight spaces of the coset up to weight 8 (with the time taken), and compares them with the integer-power coefficients of $\prod_n(1 + q^{n-1/2})$, written in the variable $t = q^{1/2}$.

In [7]:
N = 9
t0 = time.time()
dims = [C.weight_space(n).dimension() for n in range(N)]
print('weight spaces of the coset: %.1f s' % (time.time() - t0))
P.<t> = PowerSeriesRing(QQ, default_prec=2 * N)         # t = q^(1/2)
majorana = prod(1 + t^(2*n - 1) for n in range(1, N + 1))
ising = [majorana[2 * n] for n in range(N)]
dims, dims == ising

weight spaces of the coset: 0.4 s


([1, 0, 1, 1, 2, 2, 3, 3, 5], True)

**The coset conformal vector.** `C.with_conformal_vector('coset')` gives the commutant the conformal vector $T = T_V - T_A$, of central charge $2 - \frac32 = \frac12$. `strong_generators_up_to(8)` finds that up to weight 8 the coset is strongly generated by a single field of weight 2. The first cell below prints the central charge, $T$ and the weights of the strong generators; the second checks that the generator found is a multiple of $T$.

In [8]:
C = C.with_conformal_vector('coset')
T = C.conformal_vector()
gens = C.strong_generators_up_to(8)
C.central_charge(), T, [g.degree() for g in gens]

(1/2,
 -1/4*e^(-1,1) - 1/4*e^(1,-1) + 1/16*:J0 J0: - 1/8*:J0 J1: + 1/16*:J1 J1:,
 [2])

In [9]:
g, = gens
m = T.leading_support()
g * T.coefficient(m) == T * g.coefficient(m)       # g is a multiple of T

True

### The Ising null vector

The universal Virasoro vertex algebra at $c = \frac12$ has weight spaces of dimensions $1, 0, 1, 1, 2, 2, 4, 4, 7, \dots$, but the coset has only 3 states of weight 6. So the four normally ordered monomials of weight 6 in $T$,
$$\partial^4 T, \qquad {:}(\partial^2 T)\,T{:}, \qquad {:}(\partial T)(\partial T){:}, \qquad {:}TTT{:},$$
are linearly dependent in the coset: they satisfy one relation.

The cell below finds this relation. It builds the subalgebra $S$ generated by $T$, prints elements of $S$ in terms of $T$ (`print_options(generators='strong')`), and prints three things:

1. the dimensions of the weight spaces of $S$, which agree with those of the coset found above;
2. a basis of the weight-6 space of $S$, given by `basis_words()`. Each basis monomial is written as a tuple of pairs (index of the generator, number of derivatives); here the only generator is $T$, of index 0, so `((0, 4),)` is $\partial^4 T$, `((0, 2), (0, 0))` is ${:}(\partial^2 T)\,T{:}$ and `((0, 1), (0, 1))` is ${:}(\partial T)(\partial T){:}$. The cube ${:}TTT{:}$ is not in the basis;
3. the cube ${:}TTT{:}$, which is therefore printed as a combination of the three basis monomials. Read as an equation, ${:}TTT{:} = -\frac{93}{64}{:}(\partial T)(\partial T){:} + \cdots$, this is the relation.

In [10]:
S = L.subalgebra([T], names=['T'])
S.print_options(generators='strong')
Ts = S.gen('T')
TTT = Ts.no(Ts.no(Ts))
[S.weight_space(n).dimension() for n in range(N)], S.weight_space(6).basis_words(), TTT

([1, 0, 1, 1, 2, 2, 3, 3, 5],
 [((0, 4),), ((0, 2), (0, 0)), ((0, 1), (0, 1))],
 -93/64*:dT dT: + 33/16*:d^2T T: + 9/128*d^4T)

The same combination in the universal Virasoro vertex algebra at $c = \frac12$ is a singular vector: $L_1 = L_{(2)}$ and $L_2 = L_{(3)}$ kill it. It is the singular vector of weight 6 whose quotient is the Ising model $L(\frac12, 0)$ (see `06_subalgebras_ideals_quotients.ipynb`, section 5). The cell below writes the universal Virasoro algebra with its OPE (in `NormallyOrderedExpressions`, as in `01_getting_started.ipynb`, section 2), checks that the relation is singular, and that the quotient by it has the dimensions of the coset up to weight 8.

In [11]:
XL = NormallyOrderedExpressions(QQ, ['L'])         # to write the OPE before the algebra exists
l = XL.gen(0)
Vir = UniversalVertexAlgebra(QQ, XL, {('L', 'L'): [l.derivative(), 2*l, 0, 1/4]},
                             degrees={'L': 2})
Lv = Vir.gen('L')
null = Lv.no(Lv.no(Lv)) - (9/128 * Lv.derivative(4) + 33/16 * Lv.derivative(2).no(Lv)
                           - 93/64 * Lv.derivative().no(Lv.derivative()))
Q = Vir / [null]
(Lv.nproduct(null, 2), Lv.nproduct(null, 3),
 [Vir.weight_space(n).dimension() for n in range(N)],
 [Q.weight_space(n).dimension() for n in range(N)] == dims)

(0, 0, [1, 0, 1, 1, 2, 2, 4, 4, 7], True)

### The double commutant

Up to weight 8, the coset is generated by $T$, of weight 2. So `C.commutant(max_weight=2)`, the commutant of the subalgebra generated by the strong generators of `C` of weight $\le 2$, is $\mathrm{Com}(\mathrm{Com}(A, V), V)$. The cell below shows that it has the same dimensions as $A$ up to weight 5 and contains $e$, $f$ and $h$: the diagonal $L_2(\mathfrak{sl}_2)$ is itself a coset, the commutant of the Ising model, in the weights computed.

In [12]:
D = C.commutant(max_weight=2)
([D.weight_space(n).dimension() for n in range(6)],
 [A.weight_space(n).dimension() for n in range(6)],
 e in D, f in D, h in D)

([1, 3, 9, 15, 30, 54], [1, 3, 9, 15, 30, 54], True, True, True)

## 3. Kazama–Suzuki: the $N = 2$ superconformal algebra

Kazama and Suzuki [KS1989] found $N = 2$ superconformal algebras as cosets. The simplest case uses:

* the universal affine vertex algebra $V^k(\mathfrak{sl}_2)$ at a **generic** level, that is, over the field $\mathbb Q(k)$, with generators $E, H, F$ and the Sugawara vector, of central charge $3k/(k+2)$;
* a $bc$ system of two odd fields of weight $\frac12$, $b(z)c(w) \sim 1/(z-w)$ (see `04_ghost_systems.ipynb`, section 4), of central charge 1. Its fermion number current $J_f = {:}bc{:}$ gives $b$ charge $+1$ and $c$ charge $-1$.

Both algebras are written below with `UniversalVertexAlgebra` and their OPEs (`01_getting_started.ipynb`, section 2); `va.Affine` and `va.FermionicGhosts` give the same algebras. The cell prints the two central charges and the OPEs of $J_f$ with itself, with $b$ and with $c$.

In [13]:
K.<k> = Frac(QQ['k'])
XS.<E, H, F> = NormallyOrderedExpressions(K)
sl2 = UniversalVertexAlgebra(K, XS, {('H', 'H'): [0, 2*k], ('H', 'E'): [2*E], ('H', 'F'): [-2*F],
                                     ('E', 'F'): [H, k]},
                             conformal_vector=(H*H / 2 + E*F + F*E) / (2*(k + 2)))
E, H, F = sl2.gens()
XB.<b, c> = NormallyOrderedExpressions(K, parities={'b': 1, 'c': 1})
bc = UniversalVertexAlgebra(K, XB, {('b', 'c'): [1]},
                            conformal_vector=(b.derivative()*c - b*c.derivative()) / 2)
b, c = bc.gens()
Jf = b.no(c)
sl2.central_charge(), bc.central_charge(), Jf.ope(Jf), Jf.ope(b), Jf.ope(c)

(3*k/(k + 2), 1, [0, 1], [b], [-c])

In the tensor product $V = V^k(\mathfrak{sl}_2) \otimes bc$ (`tensor`, with the sum of the conformal vectors, see `05_lattice_vertex_algebras.ipynb`, section 7), the diagonal Heisenberg field $H_{\mathrm{tot}} = H + 2J_f$ has level $2k + 4$: $H_{\mathrm{tot}}(z)H_{\mathrm{tot}}(w) \sim (2k+4)/(z-w)^2$. It commutes with $E \otimes c$ and $F \otimes b$, since the charges cancel. The subalgebra $U$ it generates is given its own conformal vector $\frac{1}{2(2k+4)}{:}H_{\mathrm{tot}}H_{\mathrm{tot}}{:}$, of central charge 1. The cell below prints the OPE of $H_{\mathrm{tot}}$ and the central charges of $V$ and $U$.

In [14]:
V = sl2.tensor(bc, conformal_vector='factors')
Htot = V.embed(0, H) + 2 * V.embed(1, Jf)
U = V.subalgebra([Htot], names=['Htot'], conformal_vector=Htot.no(Htot) / (4 * (k + 2)))
Htot.ope(Htot), V.central_charge(), U.central_charge()

([0, 2*k + 4], (4*k + 2)/(k + 2), 1)

The coset $\mathrm{Com}(U, V)$ has central charge $\frac{3k}{k+2} + 1 - 1 = \frac{3k}{k+2}$. According to Kazama–Suzuki, it is the universal $N = 2$ superconformal algebra, strongly and freely generated by fields $J, G^+, G^-, T$ of weights $1, \frac32, \frac32, 2$. Its character is then
$$\prod_{n \ge 1} \frac{1}{1 - q^n} \prod_{n \ge 2} \frac{1}{1 - q^n} \prod_{r \ge 3/2} (1 + q^r)^2,$$
the product over $r \in \frac12 + \mathbb Z$. The cell below computes the dimensions of the weight spaces of the coset in the weights $0, \frac12, \dots, 5$ and compares them with this series (again in $t = q^{1/2}$).

In [15]:
C2 = V.coset(U, conformal_vector='coset')
M = 10                                                  # weights 0, 1/2, ..., 5
t0 = time.time()
dims2 = [C2.weight_space(n / 2).dimension() for n in range(M + 1)]
print('weight spaces of the coset: %.1f s' % (time.time() - t0))
P.<t> = PowerSeriesRing(QQ, default_prec=M + 1)        # t = q^(1/2)
n2 = (prod(1 / (1 - t^(2*n)) for n in range(1, M)) * prod(1 / (1 - t^(2*n)) for n in range(2, M))
      * prod((1 + t^m)^2 for m in range(3, M + 1, 2)))
C2.central_charge(), dims2, dims2 == [n2[i] for i in range(M + 1)]

weight spaces of the coset: 0.1 s


(3*k/(k + 2), [1, 0, 1, 2, 3, 4, 6, 10, 15, 20, 28], True)

**Strong generators.** Up to weight 5 the commutant is strongly generated by one field of weight 1, two of weight $\frac32$ ($F \otimes b$ and $E \otimes c$) and one of weight 2; nothing new appears in the weights $\le 5$. The cell below prints their weights and the first three. These are the canonical basis vectors of the weight spaces, not yet normalised.

In [16]:
J1, Gm1, Gp1, W1 = C2.strong_generators_up_to(5)
[g.degree() for g in C2.strong_generators_up_to(5)], J1, Gm1, Gp1

([1, 3/2, 3/2, 2], 1 # :b c: - 1/k*H # 1, F # b, E # c)

**The $N = 2$ OPEs.** The cell below normalises the generators to the standard basis of the $N = 2$ superconformal algebra:

* $J$ is rescaled so that $J(z)G^\pm(w) \sim \pm G^\pm(w)/(z-w)$;
* $T$ is the coset conformal vector;
* $G^+$ is kept and $G^-$ is rescaled so that
$$G^+(z)G^-(w) \sim \frac{2c/3}{(z-w)^3} + \frac{2J(w)}{(z-w)^2} + \frac{2T(w) + \partial J(w)}{z-w}.$$

It then checks all OPEs of the $N = 2$ superconformal algebra. They hold exactly, for generic $k$.

In [17]:
T2 = C2.conformal_vector()
cc = C2.central_charge()
m = Gp1.leading_support()
J = J1 / (J1.nproduct(Gp1, 0).coefficient(m) / Gp1.coefficient(m))
Gp = Gp1
Gm = Gm1 * (2*cc/3) / Gp1.ope(Gm1)[2].coefficient(Gp1.ope(Gm1)[2].leading_support())
checks = {
    'J J':   J.ope(J) == [0, cc/3 * C2.one()],
    'J G+':  J.ope(Gp) == [Gp],
    'J G-':  J.ope(Gm) == [-Gm],
    'T J':   T2.ope(J) == [J.derivative(), J],
    'T G+':  T2.ope(Gp) == [Gp.derivative(), 3/2 * Gp],
    'T G-':  T2.ope(Gm) == [Gm.derivative(), 3/2 * Gm],
    'G+ G-': Gp.ope(Gm) == [2*T2 + J.derivative(), 2*J, 2*cc/3 * C2.one()],
    'G+ G+': Gp.ope(Gp) == [],
    'G- G-': Gm.ope(Gm) == [],
}
checks

{'J J': True,
 'J G+': True,
 'J G-': True,
 'T J': True,
 'T G+': True,
 'T G-': True,
 'G+ G-': True,
 'G+ G+': True,
 'G- G-': True}

**The subalgebra they generate.** The cell below builds the subalgebra generated by $J, G^\pm, T$ (`06_subalgebras_ideals_quotients.ipynb`, section 1). Its closure adds no strong generator, and printed in its strong generators (`print_options(generators='strong')`) its OPEs are those of the $N = 2$ superconformal algebra. Its dimensions equal those of the coset up to weight 5, so in these weights the coset **is** the $N = 2$ superconformal algebra of central charge $c = 3k/(k+2)$.

In [18]:
N2 = V.subalgebra([J, Gp, Gm, T2], names=['J', 'G+', 'G-', 'T'])
N2.print_options(generators='strong')
Jn, Gpn, Gmn, Tn = N2.gens()
print(N2.variable_names())
for x, y in [(Gpn, Gmn), (Gmn, Gpn), (Jn, Gpn), (Tn, Gmn), (Tn, Tn)]:
    print(x, '×', y, ':', x.ope(y))
[N2.weight_space(n / 2).dimension() for n in range(M + 1)] == dims2

('J', 'G+', 'G-', 'T')
G+ × G- : [dJ + 2*T, 2*J, 2*k/(k + 2)]
G- × G+ : [-dJ + 2*T, -2*J, 2*k/(k + 2)]
J × G+ : [G+]
T × G- : [dG-, 3/2*G-]
T × T : [dT, 2*T, 0, 3/2*k/(k + 2)]


True

Elements of the coset can be written in the strong generators found by the search, named `C0, C1, ...`, within the weights searched (`express`, as for subalgebras):

In [19]:
C2.express(Gp1.no(Gm1) + J1.derivative(2))

(-1/2*k + 1)*d^2C0 - :C1 C2: - dC3

### The double commutant

Up to weight 5 the coset is strongly generated in weights $\le 2$, so `C2.commutant(max_weight=2)` is the double commutant. The cell below shows its dimensions in the weights $0, \frac12, \dots, 5$: the numbers of partitions $p(n)$ in integer weights and $0$ in half-integer weights, those of a single Heisenberg field. It contains $H_{\mathrm{tot}}$. So, in the weights computed, the Heisenberg algebra $U$ generated by $H_{\mathrm{tot}}$ is itself a coset: the commutant of the $N = 2$ superconformal algebra.

In [20]:
D2 = C2.commutant(max_weight=2)
[D2.weight_space(n / 2).dimension() for n in range(11)], Htot in D2

([1, 0, 1, 0, 2, 0, 3, 0, 5, 0, 7], True)

## 4. Limitations

* **Up to a weight.** The weight spaces of a commutant are exact, but `strong_generators_up_to(W)` makes no claim above `W`, and neither do the comparisons of dimensions above. That the GKO coset is $L(\frac12, 0)$ and the Kazama–Suzuki coset the $N = 2$ superconformal algebra in all weights are theorems [GKO1986, KS1989].
* **Commutants are not subalgebras of the library.** A commutant has no complete set of strong generators, so it has no ideals, quotients or presentation. To get these, use `V.subalgebra(C.strong_generators_up_to(W))`: it is contained in the commutant, and equality can be checked weight by weight by comparing dimensions, as in section 3.
* **Double commutants** with `max_weight=k` use the strong generators of weight $\le k$ found by `strong_generators_up_to`; the result is the double commutant only if the commutant is generated in weights $\le k$, which the computation does not prove.
* **Generic parameters.** A computation over $\mathbb Q(k)$ holds for generic $k$. At special values of $k$ the kernels can be larger, and these values must be treated separately, for example over $\mathbb Q$ at a fixed level.
* **The coset conformal vector** needs `A` as a subalgebra constructed with its conformal vector $T_A$; it is not computed from the generators of `A`.
* **Finite weight spaces.** Commutants are computed weight space by weight space, so $V$ needs finite-dimensional weight spaces (or a horizontal grading that makes the graded pieces finite).
* **Cost.** The weight spaces of $V$ grow quickly. The kernels are computed by sparse or dense linear algebra over $\mathbb Q$ or $\mathbb Q(k)$; the basis of $V$ is ordered by the key order, so the kernel comes out directly as the canonical basis, which usually has small coefficients. For example, the $\mathfrak{sl}_2$ parafermions at weight 14 (an ambient weight space of dimension 3105) take 27 s.

## References

- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X). — commutant subalgebras of vertex operator algebras.
- [GKO1985] P. Goddard, A. Kent, D. Olive, *Virasoro algebras and coset space models*, Phys. Lett. B 152 (1985), 88–92, [doi:10.1016/0370-2693(85)91145-1](https://doi.org/10.1016/0370-2693(85)91145-1). — the coset construction.
- [GKO1986] P. Goddard, A. Kent, D. Olive, *Unitary representations of the Virasoro and super-Virasoro algebras*, Comm. Math. Phys. 103 (1986), 105–119, [doi:10.1007/BF01464283](https://doi.org/10.1007/BF01464283). — the coset construction and the Ising model as a coset (section 1).
- [KS1989] Y. Kazama, H. Suzuki, *New N = 2 superconformal field theories and superstring compactification*, Nuclear Phys. B 321 (1989), 232–268, [doi:10.1016/0550-3213(89)90250-2](https://doi.org/10.1016/0550-3213(89)90250-2). — the $N=2$ superconformal algebra as a coset (section 2).
- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X). — the Ising model (section 1).